In [ ]:
import os

from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

MODEL_NAME = 'all-MiniLM-L6-v2'
LOCAL_MODELS_FOLDER = './models/'

local_path = os.path.join(LOCAL_MODELS_FOLDER, MODEL_NAME)

if not os.path.isdir(local_path):
    model = SentenceTransformer(MODEL_NAME)
    os.makedirs(local_path, exist_ok=True)
    model.save(local_path)

model = SentenceTransformer(local_path)

v_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7734.27it/s]


In [2]:
query_vector = model.encode('How do I run Kafka?')
results = vs_index.search(query_vector, num_results=5)
results

[{'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: How to run producer/consumer/kstreams/etc in terminal',
  'answer': 'In the project directory, run:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```',
  'doc_id': '5ca6890c1a'},
 {'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent',
  'answer': 'For example, when running `JsonConsumer.java`, you might see:\n\n```\nConsuming form kafka started\n\nRESULTS:::0\n\nRESULTS:::0\n\nRESULTS:::0\n```\n\nOr when running `JsonProducer.java`, you might encounter:\n\n```\nException in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed\n```\n\n**Solution:**\n\n1. Ensure the `StreamsConfig.BOOTSTRAP_SERVERS_CO

In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

from rag_helper import RAGBase

class ContextBuilder:
    def build(self, search_result):
        lines = []

        for doc in search_result:
            lines.append('question: ' + doc['question'])
            lines.append('answer: ' + doc['answer'])
            lines.append('')

        return '\n'.join(lines).strip()


class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder
    
    def search(
            self,
            query,
            boost_dict={},
            filter_dict={},
            num_results=5
        ):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=v_index,
    llm_client=openai_client,
    context_builder=ContextBuilder()
)

vector_assistant.rag('Can I join the course after it started?')


('Yes, you can still join. If you want to receive a certificate, make sure you submit your project while submissions are still open.',
 380)